# 05 · The application: daily runs, Logs.csv and recovery

Project II turned the three models into an application that runs on the plant's servers every day. It was called Central Logging at Volkswagen. This notebook runs it on a few generated days and shows how it behaves when a file is broken or a run stops halfway.

**Folders** (same as the deployed version)
```
Input_PLC/            one CSV per PLC per day, dropped in by the export
Output_PLC/           clustered_output_<file>_<run time>.csv
Input_Multi_PLC/      plant-wide files for the multi-PLC module (notebook 06)
Error_Files/          inputs that failed
scripts/Lib/Logs/Logs.csv
scripts/Lib/Processing_Files/   intermediate files and saved models
```

**Stages per file** (`src/central_logging/main.py`)

| Logs.csv column | What runs |
|---|---|
| PROCESS TIME | preparation and Model 1 |
| GENERATE EMBEDDINGS | BERT on alertText |
| PROCESS EMBEDDINGS | autoencoder and HDBSCAN (Model 2) |
| COMBINE CLUSTER | Model 3, writes the output |

Each stage writes `Passed (seconds)` or `Failed: reason` for each file. A file that passes everything is marked DONE and removed with its intermediate files. A failed file goes to `Error_Files`. When it comes back, the stages that already passed are skipped. Log entries older than 7 days are dropped.

In [1]:
import sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from central_logging import config, plotting
from central_logging.plotting import SERIES, MUTED, TEXT, TEXT_2

plotting.use_style()
pd.set_option("display.max_colwidth", 90)
RESULTS = ROOT / "results"
FIG = RESULTS / "figures"
FIG.mkdir(parents=True, exist_ok=True)

In [2]:
import shutil
import warnings
from datetime import datetime, timedelta

warnings.filterwarnings("ignore", message="IProgress not found")

from central_logging import main
from central_logging.config import Paths
from central_logging.embeddings import TextEmbedder
from central_logging.generate import alert_day, export_daily_files, generate
from central_logging.text_clusters import train_text_model

alerts, truth, plant = generate()
days = sorted(alert_day(alerts).unique())
plcs = ["A1SWE01", "A1ROB01", "A1KLB01", "A2MON01", "B1FOE01", "B1PRF01"]
demo = alerts[alerts.alertPlc.isin(plcs)]

APP = ROOT / "notebooks" / "app_demo"          # not committed
shutil.rmtree(APP, ignore_errors=True)
app = Paths(APP).make()
embedder = TextEmbedder()

## Before the first run: the text model
The scaler, autoencoder and HDBSCAN come from the history (here: the first week of these six PLCs). The autoencoder is trained on the first PLC and then updated PLC by PLC, as in Project I.

In [3]:
history = demo[alert_day(demo).isin(days[:7])]
sets = [(plc, embedder.embed(g.alertText.tolist())) for plc, g in history.groupby("alertPlc")]
training, _ = train_text_model(sets, app.text_model)
training.round(4)

,plc_file,rows,first_epoch_loss,last_epoch_loss
0,A1KLB01,514,0.0586,0.0162
1,A1ROB01,424,0.2308,0.0033
2,A1SWE01,503,0.2366,0.0048
3,A2MON01,805,0.3036,0.0319
4,B1FOE01,349,0.4822,0.0043
5,B1PRF01,229,0.3213,0.0018


## Day 1, with one broken file

In [4]:
def run_day(day, **kwargs):
    return main.run(APP, embedder=embedder, now=datetime.fromisoformat(day) + timedelta(days=1, hours=5), **kwargs)

def ledger():
    return pd.read_csv(app.ledger)

day1, day2 = days[7], days[8]
export_daily_files(demo, day1, single_dir=app.input)
broken = pd.read_csv(app.input / f"A1ROB01_{day1.replace('-', '')}.csv")
broken.loc[5, "alertBegin"] = "2024-04-15 25:61:00"          # an impossible timestamp
broken.to_csv(app.input / f"A1ROB01_{day1.replace('-', '')}.csv", index=False)

run_day(day1)

INFO: Starting central logging pipeline at 2024-04-16 05:00:00


INFO: Found 6 files, backup in Input_PLC/backup/20240416_050000


ERROR: PROCESS TIME failed on A1ROB01_20240415.csv: time data "2024-04-15 25:61:00" doesn't match format "%Y-%m-%d %H:%M:%S.%f %z". You might want to try:


INFO: PROCESS TIME finished for 5 of 6 files


INFO: GENERATE EMBEDDINGS finished for 5 of 6 files


INFO: PROCESS EMBEDDINGS finished for 5 of 6 files


INFO: COMBINE CLUSTER finished for 5 of 6 files


INFO: Run complete: 5 passed, 1 failed


,TO DO,PROCESS TIME,GENERATE EMBEDDINGS,PROCESS EMBEDDINGS,COMBINE CLUSTER,DONE,TIMESTAMP
0,A1KLB01_20240415,Passed (0.06s),Passed (0.00s),Passed (0.03s),Passed (0.03s),Passed,2024-04-16 05:00:00
1,A1ROB01_20240415,"Failed: time data ""2024-04-15 25:61:00"" doesn't match format ""%Y-%m-%d %H:%M:%S.%f %z""...",NaN,NaN,NaN,Failed,2024-04-16 05:00:00
2,A1SWE01_20240415,Passed (0.04s),Passed (0.00s),Passed (0.01s),Passed (0.01s),Passed,2024-04-16 05:00:00
3,A2MON01_20240415,Passed (0.05s),Passed (0.09s),Passed (0.01s),Passed (0.01s),Passed,2024-04-16 05:00:00
4,B1FOE01_20240415,Passed (0.03s),Passed (0.00s),Passed (0.00s),Passed (0.01s),Passed,2024-04-16 05:00:00
5,B1PRF01_20240415,Passed (0.04s),Passed (0.00s),Passed (0.00s),Passed (0.01s),Passed,2024-04-16 05:00:00


In [5]:
print("Input_PLC:  ", sorted(p.name for p in app.input.glob("*.csv")))
print("Error_Files:", sorted(p.name for p in app.errors.glob("*.csv")))
print("Output_PLC: ", len(list(app.output.glob("*.csv"))), "files")

Input_PLC:   []
Error_Files: ['A1ROB01_20240415.csv']
Output_PLC:  5 files


The broken file failed in the first stage and was moved to `Error_Files`. The other five went through. After fixing the timestamp and moving the file back, the next run picks it up again:

In [6]:
name = f"A1ROB01_{day1.replace('-', '')}.csv"
fixed = demo[(alert_day(demo) == day1) & (demo.alertPlc == "A1ROB01")]
fixed.to_csv(app.input / name, index=False)
(app.errors / name).unlink()
run_day(day1)

INFO: Starting central logging pipeline at 2024-04-16 05:00:00


INFO: Found 1 files, backup in Input_PLC/backup/20240416_050000


INFO: PROCESS TIME finished for 1 of 1 files


INFO: GENERATE EMBEDDINGS finished for 1 of 1 files


INFO: PROCESS EMBEDDINGS finished for 1 of 1 files


INFO: COMBINE CLUSTER finished for 1 of 1 files


INFO: Run complete: 1 passed, 0 failed


,TO DO,PROCESS TIME,GENERATE EMBEDDINGS,PROCESS EMBEDDINGS,COMBINE CLUSTER,DONE,TIMESTAMP
1,A1ROB01_20240415,Passed (0.04s),Passed (0.00s),Passed (0.02s),Passed (0.02s),Passed,2024-04-16 05:00:00


## Day 2, stopped halfway and resumed

In [7]:
export_daily_files(demo, day2, single_dir=app.input)
run_day(day2, stop_after="GENERATE EMBEDDINGS")

INFO: Starting central logging pipeline at 2024-04-17 05:00:00


INFO: Found 6 files, backup in Input_PLC/backup/20240417_050000


INFO: PROCESS TIME finished for 6 of 6 files


INFO: GENERATE EMBEDDINGS finished for 6 of 6 files


,TO DO,PROCESS TIME,GENERATE EMBEDDINGS,PROCESS EMBEDDINGS,COMBINE CLUSTER,DONE,TIMESTAMP
6,A1KLB01_20240416,Passed (0.05s),Passed (0.00s),NaN,NaN,NaN,2024-04-17 05:00:00
7,A1ROB01_20240416,Passed (0.04s),Passed (0.07s),NaN,NaN,NaN,2024-04-17 05:00:00
8,A1SWE01_20240416,Passed (0.05s),Passed (0.00s),NaN,NaN,NaN,2024-04-17 05:00:00
9,A2MON01_20240416,Passed (0.05s),Passed (0.00s),NaN,NaN,NaN,2024-04-17 05:00:00
10,B1FOE01_20240416,Passed (0.04s),Passed (0.00s),NaN,NaN,NaN,2024-04-17 05:00:00
11,B1PRF01_20240416,Passed (0.07s),Passed (0.07s),NaN,NaN,NaN,2024-04-17 05:00:00


In [8]:
before = ledger().set_index("TO DO")
run_day(day2)
after = ledger().set_index("TO DO")
d2 = [i for i in after.index if i.endswith(day2.replace("-", ""))]
pd.DataFrame({"PROCESS TIME, first run": before.loc[d2, "PROCESS TIME"],
              "PROCESS TIME, after resume": after.loc[d2, "PROCESS TIME"],
              "COMBINE CLUSTER": after.loc[d2, "COMBINE CLUSTER"], "DONE": after.loc[d2, "DONE"]})

INFO: Starting central logging pipeline at 2024-04-17 05:00:00


INFO: Found 6 files, backup in Input_PLC/backup/20240417_050000


INFO: PROCESS TIME finished for 6 of 6 files


INFO: GENERATE EMBEDDINGS finished for 6 of 6 files


INFO: PROCESS EMBEDDINGS finished for 6 of 6 files


INFO: COMBINE CLUSTER finished for 6 of 6 files


INFO: Run complete: 6 passed, 0 failed


,"PROCESS TIME, first run","PROCESS TIME, after resume",COMBINE CLUSTER,DONE
TO DO,,,,
A1KLB01_20240416,Passed (0.05s),Passed (0.05s),Passed (0.02s),Passed
A1ROB01_20240416,Passed (0.04s),Passed (0.04s),Passed (0.02s),Passed
A1SWE01_20240416,Passed (0.05s),Passed (0.05s),Passed (0.01s),Passed
A2MON01_20240416,Passed (0.05s),Passed (0.05s),Passed (0.02s),Passed
B1FOE01_20240416,Passed (0.04s),Passed (0.04s),Passed (0.01s),Passed
B1PRF01_20240416,Passed (0.07s),Passed (0.07s),Passed (0.01s),Passed


The times in the first two columns are identical: the resumed run did not repeat the stages that had already passed.

## What an output file looks like

In [9]:
path = sorted(app.output.glob("clustered_output_A1SWE01_*"))[-1]
out = pd.read_csv(path)
print(path.name, f"{len(out)} alerts, {int((out.Cluster >= 0).sum())} of them in a Model 1 group")
cols = ["alertId", "alertBegin", "alertText", "alertIsBandStop", "Cluster", "alert_combined",
        "Text_Cluster", "Final_Cluster"]
out.sort_values("alertBegin")[cols].head(10)

clustered_output_A1SWE01_20240416_20261007_041920.csv 73 alerts, 0 of them in a Model 1 group


,alertId,alertBegin,alertText,alertIsBandStop,Cluster,alert_combined,Text_Cluster,Final_Cluster
0,10100189,2024-04-15 22:23:43.360863 +00:00,SPN9 Spanner 1 Endlage zu fehlt,0,-1,12,-1,3
1,10100251,2024-04-15 22:34:19.482423 +00:00,AB5 Absaugung 16 Unterdruck fehlt,0,-1,12,47,1
2,10100419,2024-04-15 22:51:31.845834 +00:00,SPN9 Spanner 16 Endlage auf fehlt,0,-1,12,58,1
3,10100491,2024-04-15 22:59:29.960514 +00:00,WW9 Werkzeugwechsler 3 ohne Quittung,1,-1,12,54,7
4,10100657,2024-04-15 23:25:47.730844 +00:00,WW9 Werkzeugwechsler 3 ohne Quittung,0,-1,13,54,1
5,10100876,2024-04-15 23:45:37.153673 +00:00,SPN9 Spanner 16 Endlage auf fehlt,0,-1,13,58,1
6,10101073,2024-04-16 00:06:44.918432 +00:00,RB7 Roboter 17 nicht in Grundstellung,0,-1,0,39,3
7,10101103,2024-04-16 00:11:33.102685 +00:00,AL5 Bauteil 11 liegt nicht auf,0,-1,0,68,0
8,10102270,2024-04-16 02:32:50.996712 +00:00,LV9 Lichtvorhang 19 unterbrochen,1,-1,14,-1,7
9,10102387,2024-04-16 02:50:16.227654 +00:00,WW9 Werkzeugwechsler 3 ohne Quittung,0,-1,14,54,1


## Log retention
When the next files arrive more than a week later, the run first drops the ledger rows that are older than 7 days.

In [10]:
rows_before = len(ledger())
day3 = days[9]
export_daily_files(demo, day3, single_dir=app.input)
main.run(APP, embedder=embedder, now=datetime.fromisoformat(day2) + timedelta(days=9))
print(f"Logs.csv rows: {rows_before} before the run; after it, {len(ledger())} "
      f"(the {len(plcs)} new rows of {day3} only)")

INFO: Starting central logging pipeline at 2024-04-25 00:00:00


INFO: Found 6 files, backup in Input_PLC/backup/20240425_000000


INFO: Removed 12 ledger rows older than 7 days


INFO: PROCESS TIME finished for 6 of 6 files


INFO: GENERATE EMBEDDINGS finished for 6 of 6 files


INFO: PROCESS EMBEDDINGS finished for 6 of 6 files


INFO: COMBINE CLUSTER finished for 6 of 6 files


INFO: Run complete: 6 passed, 0 failed


Logs.csv rows: 12 before the run; after it, 6 (the 6 new rows of 2024-04-17 only)


## Takeaway
- One broken file does not stop the run. It is recorded, set aside, and processed once it is fixed.
- A stopped run resumes where it ended, so a crash during a long embedding step does not cost the earlier work.
- Models are loaded and updated, never retrained, so a daily run takes seconds per file.